In [1]:
"""Write stories with your trained model. No training, no dataset needed.
Needs in the same folder: slm_final.pt  and  data/tokenizer.json
  python generate.py                         # interactive
  python generate.py "The old man walked"    # one-shot
"""
import math
import sys
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

In [2]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if dev == "cuda" else torch.float32
 
ck = torch.load("slm_final.pt", map_location=dev)
c = ck["config"]
VOCAB, DIM, LAYERS, HEADS, FF, CTX = (c[k] for k in ("VOCAB", "DIM", "LAYERS", "HEADS", "FF", "CTX"))
 
 
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d)); self.eps = eps
 
    def forward(self, x):
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32 * self.w).type_as(x)
 
 
def apply_rope(x, cos, sin):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)
 
 
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(DIM, 3 * DIM, bias=False)
        self.proj = nn.Linear(DIM, DIM, bias=False)
 
    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, HEADS, DIM // HEADS).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, DIM))
 
 
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.gate = nn.Linear(DIM, FF, bias=False)
        self.up = nn.Linear(DIM, FF, bias=False)
        self.down = nn.Linear(FF, DIM, bias=False)
 
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))
 
 
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.n1, self.attn = RMSNorm(DIM), Attention()
        self.n2, self.mlp = RMSNorm(DIM), MLP()
 
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))
 
 
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, DIM)
        self.blocks = nn.ModuleList(Block() for _ in range(LAYERS))
        self.norm = RMSNorm(DIM)
        self.head = nn.Linear(DIM, VOCAB, bias=False)
        self.head.weight = self.emb.weight
        hd = DIM // HEADS
        inv = 1.0 / (10000 ** (torch.arange(0, hd, 2).float() / hd))
        f = torch.outer(torch.arange(CTX).float(), inv)
        self.register_buffer("cos", f.cos(), persistent=False)
        self.register_buffer("sin", f.sin(), persistent=False)
 
    def forward(self, idx):
        T = idx.shape[1]
        x = self.emb(idx)
        for b in self.blocks:
            x = b(x, self.cos[:T], self.sin[:T])
        return self.head(self.norm(x))
 
    @torch.no_grad()
    def generate(self, idx, n, temp, top_k=50):
        for _ in range(n):
            logits = self(idx[:, -CTX:])[:, -1].float() / temp
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
        return idx


In [ ]:
tok = Tokenizer.from_file("data/tokenizer.json")
model = GPT().to(dev)
model.load_state_dict(ck["model"])
model.eval()
 
 
def write(prompt, n=250, temp=0.8):
    ids = torch.tensor([tok.encode(prompt).ids], device=dev)
    with torch.autocast(dev, dtype=dtype, enabled=dev == "cuda"):
        out = model.generate(ids, n, temp)
    return tok.decode(out[0].tolist())
 
 
def ask_number(msg, default, cast):
    raw = input(msg).strip()
    try:
        return cast(raw) if raw else default
    except ValueError:
        print(f"  (not a number, using {default})")
        return default
 
 
if __name__ == "__main__":
    print("=" * 60)
    print(" Story writer  |  14M-parameter model trained from scratch")
    print("=" * 60)
    print("You write the beginning of a story, and the model continues it.")
    print("Tip: 2-3 sentences with your characters and setting work best.")
    print("Type 'quit' to exit.\n")
 
    temp, length = 0.8, 250
    while True:
        beginning = input("Write the beginning of your story:\n> ").strip()
        if beginning.lower() in ("quit", "exit", "q"):
            print("Bye!")
            break
        if not beginning:
            print("  Please type at least a few words.\n")
            continue
 
        length = ask_number(f"How long? tokens, ~0.7 words each [{length}]: ", length, int)
        length = max(20, min(length, CTX - 20))
        temp = ask_number(f"Creativity 0.5 (safe) to 1.1 (wild) [{temp}]: ", temp, float)
        temp = max(0.3, min(temp, 1.5))
 
        print("\nWriting...\n")
        print(write(beginning, n=length, temp=temp))
        print("\n" + "-" * 60 + "\n")


 Story writer  |  14M-parameter model trained from scratch
You write the beginning of a story, and the model continues it.
Tip: 2-3 sentences with your characters and setting work best.
Type 'quit' to exit.



Write the beginning of your story:
>  Once upon a time, Anuj the nigger
How long? tokens, ~0.7 words each [250]:  200
Creativity 0.5 (safe) to 1.1 (wild) [0.8]:  0.7



Writing...

Once upon a time, Anuj the nigger dinosaurs could be so funny. They had a lot of friends. One day, they heard a loud roar. A big dinosaur was stuck under a tree! "Help!" he called.

"What should we do?" Mia asked. "We have just five minutes!" said the dinosaur. "Let's help!" said Mia. They ran to the tree. The T-Rex looked at them, surprised. "I could not get a snack here," said the T-Rex. Mia thought hard. "We can help!" 

They took off their shoes and walked through the trees. The T-Rex was very big. "Can you carry that rock?" he asked. "Yes!" Mia replied. They got to the tree and placed the rock on the tree. It turned out to be a small rock! The T-Rex was so happy now. 

"Thank you, Mia!" said the T-Rex. "You are very kind!" Mia smiled. "We

------------------------------------------------------------

